# BeeWatch — Audio Autoencoder Training
**Bagian:** Model Audio  
**Framework:** TensorFlow / Keras  
**Dataset:** Kaggle `annajyang/beehive-sounds` (file WAV)  
**Fitur input:** 72 dimensi akustik (MFCC + Mel-Spectrogram + Spectral)

---
### Alur Notebook
1. Setup & import
2. Load cache fitur audio (skip ekstraksi ulang)
3. Normalisasi (StandardScaler, fit hanya di train)
4. Bangun & latih Audio Autoencoder (Keras)
5. Hitung reconstruction error & threshold
6. Visualisasi & evaluasi
7. Simpan artefak


## 1. Setup & Import

In [ ]:
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)
os.makedirs('models', exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")


## 2. Load Cache Fitur Audio

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cache berisi fitur 72-dim yang sudah diekstrak dari WAV
# Tidak perlu ekstraksi ulang — hemat waktu ~30 menit
# ─────────────────────────────────────────────────────────────

# Coba path Kaggle dataset dulu, fallback ke working dir
CACHE_PATHS = [
    '/kaggle/input/beewatch-cache/audio_features_cache.pkl',
    '/kaggle/working/audio_features_cache.pkl',
    'audio_features_cache.pkl',
]

cache = None
for path in CACHE_PATHS:
    if os.path.exists(path):
        cache = joblib.load(path)
        print(f"Cache loaded dari: {path}")
        break

if cache is None:
    raise FileNotFoundError(
        "Cache tidak ditemukan! Upload audio_features_cache.pkl "
        "sebagai Kaggle dataset lalu attach ke notebook ini."
    )

# Cache berupa list of dict: [{'filename': str, 'features': np.array(72,)}, ...]
feature_matrix = np.vstack([item['features'] for item in cache])
filenames      = [item['filename'] for item in cache]

print(f"Total sampel audio : {len(feature_matrix)}")
print(f"Dimensi fitur      : {feature_matrix.shape[1]}")
print(f"Contoh filename    : {filenames[0]}")


## 3. Eksplorasi Fitur Audio

In [ ]:
print("=== Statistik Fitur Audio (72 dimensi) ===")
print(f"  Mean per fitur — min: {feature_matrix.mean(axis=0).min():.3f}, "
      f"max: {feature_matrix.mean(axis=0).max():.3f}")
print(f"  Std  per fitur — min: {feature_matrix.std(axis=0).min():.3f}, "
      f"max: {feature_matrix.std(axis=0).max():.3f}")
print(f"  NaN count      : {np.isnan(feature_matrix).sum()}")
print(f"  Inf count      : {np.isinf(feature_matrix).sum()}")

# Drop NaN/Inf jika ada
mask           = ~(np.isnan(feature_matrix).any(axis=1) | np.isinf(feature_matrix).any(axis=1))
feature_matrix = feature_matrix[mask]
print(f"\nSetelah drop NaN/Inf: {feature_matrix.shape[0]} sampel")


In [ ]:
# Visualisasi distribusi 12 fitur pertama (MFCC mean)
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
feature_labels = (
    [f'MFCC-{i} mean' for i in range(13)] +
    [f'MFCC-{i} std'  for i in range(13)] +
    [f'Mel-{i} mean'  for i in range(20)] +
    [f'Mel-{i} std'   for i in range(20)] +
    ['SC mean', 'SC std', 'SRF mean', 'SRF std', 'ZCR mean', 'ZCR std']
)
for i in range(12):
    axes[i].hist(feature_matrix[:, i], bins=40, color='steelblue',
                 alpha=0.8, edgecolor='white')
    axes[i].set_title(feature_labels[i], fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Distribusi 12 Fitur Audio Pertama (MFCC Mean)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('audio_feature_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# PCA 2D sebelum normalisasi
pca_raw = PCA(n_components=2)
feat_2d = pca_raw.fit_transform(feature_matrix)

plt.figure(figsize=(8, 5))
plt.scatter(feat_2d[:, 0], feat_2d[:, 1], alpha=0.3, s=8, color='steelblue')
plt.xlabel(f'PC1 ({pca_raw.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_raw.explained_variance_ratio_[1]:.1%})')
plt.title('PCA Audio Features — Sebelum Normalisasi')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('audio_pca_raw.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Variansi dijelaskan PC1+PC2: "
      f"{pca_raw.explained_variance_ratio_[:2].sum():.1%}")


## 4. Persiapan Data — Split & Normalisasi

In [ ]:
# ─────────────────────────────────────────────────────────────
# PENTING: Split DULU, baru fit scaler hanya di train
# ─────────────────────────────────────────────────────────────
X_train_raw, X_val_raw = train_test_split(
    feature_matrix, test_size=0.20, random_state=42
)
print(f"Train : {X_train_raw.shape[0]} sampel")
print(f"Val   : {X_val_raw.shape[0]} sampel")

audio_scaler = StandardScaler()
X_train = audio_scaler.fit_transform(X_train_raw)  # fit hanya di train
X_val   = audio_scaler.transform(X_val_raw)         # transform saja di val

print(f"\nSetelah normalisasi (train):")
print(f"  Mean : {X_train.mean(axis=0).mean():.4f}  (~0)")
print(f"  Std  : {X_train.std(axis=0).mean():.4f}   (~1)")

joblib.dump(audio_scaler, 'models/audio_scaler.pkl')
print("\n✅ audio_scaler.pkl disimpan")


## 5. Arsitektur Model

In [ ]:
# Arsitektur: 72 → 64 → 32 (bottleneck) → 64 → 72
# Progressive compression — tidak expand dulu

AUDIO_INPUT_DIM = feature_matrix.shape[1]  # 72
ENCODING_DIM    = 32

def build_audio_autoencoder(input_dim: int, encoding_dim: int = 32) -> keras.Model:
    inp = keras.Input(shape=(input_dim,), name='audio_input')

    # Encoder
    x = layers.Dense(64, name='enc_dense1')(inp)
    x = layers.BatchNormalization(momentum=0.99, epsilon=0.001, name='enc_bn1')(x)
    x = layers.Activation('relu', name='enc_relu1')(x)
    x = layers.Dropout(0.1, name='enc_drop1')(x)
    encoded = layers.Dense(encoding_dim, activation='relu', name='latent')(x)

    # Decoder
    x = layers.Dense(64, name='dec_dense1')(encoded)
    x = layers.BatchNormalization(momentum=0.99, epsilon=0.001, name='dec_bn1')(x)
    x = layers.Activation('relu', name='dec_relu1')(x)
    x = layers.Dropout(0.1, name='dec_drop1')(x)
    decoded = layers.Dense(input_dim, activation='linear', name='audio_output')(x)

    return keras.Model(inputs=inp, outputs=decoded, name='AudioAutoencoder')


audio_model = build_audio_autoencoder(AUDIO_INPUT_DIM, ENCODING_DIM)
audio_model.summary()


## 6. Training

In [ ]:
audio_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

cbs = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=10, min_lr=1e-6, verbose=1
    )
]

print("Memulai training...")
history = audio_model.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=250,
    batch_size=64,
    callbacks=cbs,
    verbose=1
)

final_epoch = len(history.history['loss'])
print(f"\nTraining selesai di epoch : {final_epoch}")
print(f"Final train loss          : {history.history['loss'][-1]:.6f}")
print(f"Final val loss            : {history.history['val_loss'][-1]:.6f}")
print(f"Best val loss             : {min(history.history['val_loss']):.6f}")


## 7. Visualisasi Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history.history['loss'],     label='Train Loss',
             color='steelblue', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',
             color='tomato', linewidth=2)
best_ep = int(np.argmin(history.history['val_loss']))
axes[0].axvline(best_ep, color='gray', linestyle=':', alpha=0.7,
                label=f'Best epoch {best_ep+1}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curve
axes[1].plot(history.history['mae'],     label='Train MAE',
             color='steelblue', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE',
             color='tomato', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training & Validation MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Audio Autoencoder — Kurva Training', fontsize=14)
plt.tight_layout()
plt.savefig('audio_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Hitung Reconstruction Error & Threshold

In [ ]:
# Hitung reconstruction error di data TRAINING
X_train_recon = audio_model.predict(X_train, verbose=0)
train_errors  = np.mean((X_train - X_train_recon) ** 2, axis=1)

# Hitung di val
X_val_recon = audio_model.predict(X_val, verbose=0)
val_errors  = np.mean((X_val - X_val_recon) ** 2, axis=1)

print("=== Reconstruction Error (Training) ===")
print(f"  Mean : {train_errors.mean():.6f}")
print(f"  Std  : {train_errors.std():.6f}")
print(f"  P95  : {np.percentile(train_errors, 95):.6f}")
print(f"  P99  : {np.percentile(train_errors, 99):.6f}")

print("\n=== Reconstruction Error (Validation) ===")
print(f"  Mean : {val_errors.mean():.6f}")
print(f"  P95  : {np.percentile(val_errors, 95):.6f}")

# Threshold dari P95 training
audio_threshold = float(np.percentile(train_errors, 95))
audio_mean      = float(train_errors.mean())
audio_std       = float(train_errors.std())

print(f"\n>>> Threshold P95 : {audio_threshold:.6f}")
print(f"    Mean          : {audio_mean:.6f}")
print(f"    Std           : {audio_std:.6f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi error
axes[0].hist(train_errors, bins=60, alpha=0.7, color='steelblue',
             edgecolor='white', label='Train')
axes[0].hist(val_errors,   bins=60, alpha=0.5, color='seagreen',
             edgecolor='white', label='Val')
axes[0].axvline(audio_threshold, color='red', linestyle='--',
                linewidth=2, label=f'Threshold P95 = {audio_threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Reconstruction Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error terurut
axes[1].scatter(range(len(train_errors)), np.sort(train_errors),
                s=3, alpha=0.4, color='steelblue')
axes[1].axhline(audio_threshold, color='red', linestyle='--',
                linewidth=2, label=f'Threshold = {audio_threshold:.4f}')
axes[1].set_xlabel('Sampel (diurutkan)')
axes[1].set_ylabel('Reconstruction Error')
axes[1].set_title('Error Terurut — Train')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Audio Autoencoder — Analisis Reconstruction Error', fontsize=14)
plt.tight_layout()
plt.savefig('audio_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

anomaly_rate_train = (train_errors > audio_threshold).mean() * 100
anomaly_rate_val   = (val_errors   > audio_threshold).mean() * 100
print(f"Anomaly rate train : {anomaly_rate_train:.1f}% (target: ~5%)")
print(f"Anomaly rate val   : {anomaly_rate_val:.1f}%")


## 9. Visualisasi & Evaluasi

In [ ]:
# ── Sensitivitas Threshold ──────────────────────────────────
print("Analisis Sensitivitas Threshold:")
print(f"{'Persentil':>10} {'Threshold':>12} {'Anomali (n)':>12} {'Anomali (%)':>12}")
print('-' * 50)
for pct in [90, 92, 95, 97, 98, 99]:
    thr_val = float(np.percentile(train_errors, pct))
    n_anom  = int((train_errors > thr_val).sum())
    marker  = ' ← dipilih' if pct == 95 else ''
    print(f"{pct:>10} {thr_val:>12.6f} {n_anom:>12} "
          f"{n_anom/len(train_errors)*100:>11.1f}%{marker}")


In [ ]:
# ── Boxplot Train vs Val ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(
    [train_errors, val_errors],
    labels=['Train', 'Val'],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue', color='steelblue'),
    medianprops=dict(color='red', linewidth=2)
)
axes[0].axhline(audio_threshold, color='red', linestyle='--',
                linewidth=1.5, label=f'Threshold = {audio_threshold:.4f}')
axes[0].set_ylabel('Reconstruction Error')
axes[0].set_title('Error Distribution: Train vs Val')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Score distribution
upper       = audio_mean + 3 * audio_std
all_raw     = np.concatenate([train_errors, val_errors])
all_scores  = np.clip((all_raw - audio_mean) / (upper - audio_mean + 1e-8), 0, 1)
thresh_score = float(np.percentile(all_scores, 95))

axes[1].hist(all_scores, bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(thresh_score, color='red', linestyle='--',
                linewidth=2, label=f'Threshold score = {thresh_score:.4f}')
axes[1].set_xlabel('Audio Anomaly Score (0–1)')
axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Distribusi Audio Anomaly Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Audio Autoencoder — Evaluasi', fontsize=14)
plt.tight_layout()
plt.savefig('audio_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Bottleneck PCA ──────────────────────────────────────────
encoder_model = keras.Model(
    inputs=audio_model.input,
    outputs=audio_model.get_layer('latent').output
)

X_all_norm    = audio_scaler.transform(feature_matrix)
latent_repr   = encoder_model.predict(X_all_norm, batch_size=64, verbose=0)
all_errors_all = np.mean(
    (X_all_norm - audio_model.predict(X_all_norm, batch_size=64, verbose=0)) ** 2,
    axis=1
)

pca_latent = PCA(n_components=2)
latent_2d  = pca_latent.fit_transform(latent_repr)

plt.figure(figsize=(9, 6))
sc = plt.scatter(latent_2d[:, 0], latent_2d[:, 1],
                 c=all_errors_all, cmap='YlOrRd', alpha=0.5, s=10)
plt.colorbar(sc, label='Reconstruction Error')
plt.xlabel(f'PC1 ({pca_latent.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_latent.explained_variance_ratio_[1]:.1%})')
plt.title('Representasi Bottleneck (32D → PCA 2D)\nWarna = Reconstruction Error')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('audio_bottleneck_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Variansi PC1+PC2: {pca_latent.explained_variance_ratio_[:2].sum():.1%}")


In [ ]:
# ── Ringkasan Evaluasi ──────────────────────────────────────
print('=' * 55)
print(f"{'RINGKASAN EVALUASI AUDIO AUTOENCODER':^55}")
print('=' * 55)
print(f"{'Metrik':<35} {'Nilai':>18}")
print('-' * 55)
print(f"{'Training samples':<35} {len(X_train):>18,}")
print(f"{'Validation samples':<35} {len(X_val):>18,}")
print(f"{'Feature dimension':<35} {AUDIO_INPUT_DIM:>18}")
print(f"{'Encoding dimension':<35} {ENCODING_DIM:>18}")
print(f"{'Epochs trained':<35} {final_epoch:>18}")
print(f"{'Best val loss':<35} {min(history.history['val_loss']):>18.6f}")
print(f"{'Train error mean':<35} {train_errors.mean():>18.6f}")
print(f"{'Train error std':<35} {train_errors.std():>18.6f}")
print(f"{'Threshold (P95)':<35} {audio_threshold:>18.6f}")
print(f"{'Anomaly rate train':<35} {anomaly_rate_train:>17.1f}%")
print(f"{'Anomaly rate val':<35} {anomaly_rate_val:>17.1f}%")


## 10. Simpan Artefak

In [ ]:
# 1. Model
audio_model.save('models/audio_autoencoder.keras')
print("✅ Model     : models/audio_autoencoder.keras")

# 2. Scaler (sudah disimpan di Sel 4)
print("✅ Scaler    : models/audio_scaler.pkl")

# 3. Thresholds
thresholds = {
    'audio_threshold_pct95': audio_threshold,
    'audio_mean'            : audio_mean,
    'audio_std'             : audio_std,
    'combined_threshold'    : audio_threshold,
    'w_audio'               : 1.0,
    'w_sensor'              : 0.0,
}
joblib.dump(thresholds, 'models/thresholds.pkl')
print("✅ Thresholds: models/thresholds.pkl")

# 4. Model config
model_config = {
    'audio_input_dim' : AUDIO_INPUT_DIM,
    'audio_encoding'  : ENCODING_DIM,
    'audio_arch'      : [64, ENCODING_DIM],
    'audio_dropout'   : 0.1,
    'sr'              : 16000,
    'segment_duration': 2.0,
    'n_mfcc'          : 13,
    'n_mels'          : 20,
    'n_fft'           : 512,
    'hop_length'      : 256,
    'version'         : 'v3_audio_only',
}
joblib.dump(model_config, 'models/model_config.pkl')
print("✅ Config    : models/model_config.pkl")

print()
print("File yang di-push ke GitHub:")
for f in ['models/audio_autoencoder.keras', 'models/audio_scaler.pkl',
          'models/thresholds.pkl', 'models/model_config.pkl']:
    print(f"  {f}")


## 11. Verifikasi — Simulasi Inferensi

In [ ]:
import librosa

loaded_model   = tf.keras.models.load_model('models/audio_autoencoder.keras')
loaded_scaler  = joblib.load('models/audio_scaler.pkl')
loaded_thr     = joblib.load('models/thresholds.pkl')
loaded_cfg     = joblib.load('models/model_config.pkl')

# Simulasi dengan fitur dummy (rata-rata dari training)
dummy_feat   = feature_matrix[:10].mean(axis=0).reshape(1, -1)
dummy_scaled = loaded_scaler.transform(dummy_feat)
dummy_recon  = loaded_model.predict(dummy_scaled, verbose=0)
dummy_err    = float(np.mean((dummy_scaled - dummy_recon) ** 2))

upper = loaded_thr['audio_mean'] + 3 * loaded_thr['audio_std']
score = float(np.clip(
    (dummy_err - loaded_thr['audio_mean']) / (upper - loaded_thr['audio_mean'] + 1e-8),
    0, 1
))

print("=== Verifikasi Load Artefak ===")
print(f"  Model load    : OK")
print(f"  Scaler load   : OK")
print(f"  Threshold     : {loaded_thr['audio_threshold_pct95']:.6f}")
print()
print("=== Simulasi Inferensi (fitur rata-rata training) ===")
print(f"  Raw error     : {dummy_err:.6f}")
print(f"  Audio score   : {score:.4f}")
print(f"  Anomali?      : {'YA ⚠️' if score > loaded_thr['audio_threshold_pct95'] else 'TIDAK ✅'}")
